# Wake Word Quickstart — Kaggle / Colab

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Trains a wake-word ONNX model with wakeforge (imported as `ww_trainer`) end to end: dataset generation, training, export, and loading the result through the OVOS wake-word plugin.

| Field | Value |
|---|---|
| Inputs | a wake phrase (`WAKE_WORD`); optionally a folder of real recordings, or an existing dataset directory (`DATASET_DIR`) built by `synthetic_ww_dataset.ipynb` |
| Outputs | `best_f1_featurizer.onnx` and `best_f1.onnx` under `OUTPUT_DIR/model` |
| Runtime | GPU: not measured. CPU smoke path (`SMOKE=True`, `N_SAMPLES` synthetic samples, 1 epoch): a few minutes |
| Hardware | GPU (T4 or better) for a usable model; CPU is enough for the smoke path |
| Cost | GPU (T4 or better); not measured for a full run |

CI: linted, compiled under Python 3.11 and 3.12, and executed on the smoke path.

**Read more:**

- [Wake word plugins](https://openvoiceos.github.io/ovos-technical-manual/312-wake_word_plugins/)
- [wakeforge](https://github.com/TigreGotico/wakeforge)
- [ovos-ww-plugin-wakeforge](https://github.com/OpenVoiceOS/ovos-ww-plugin-wakeforge)

In [ ]:
SMOKE = False
N_SAMPLES = 32
EPOCHS = 1
SMOKE_WAKE_WORD = "hey_jarvis"  # wake word slug for this notebook's own smoke dataset; not a model id

## Cell 2 — Configuration

**This is the only cell you need to edit.**

Change `WAKE_WORD` to your phrase and click **Run All**.  Everything else has sensible
defaults for a first run on Kaggle GPU T4.

Every variable can also be provided as an environment variable or a Kaggle Secret with the
same name — `os.environ.get(...)` picks those up automatically.

> **Language tip:** wakeforge (imported as `ww_trainer`) discovers every installed
> OVOS TTS plugin and, per synthesised sample, picks one at random and calls it with
> `lang=WW_LANG` — there is no per-sample voice override, and no `DatagenConfig` field
> takes an explicit plugin or voice. With only `ovos-tts-plugin-edge-tts` installed
> (the default from `wakeforge[datagen]`) and the default `WW_LANG="en"` (no region
> tag), edge-tts's own language table has no `"en"` entry, so it falls back to its
> class default voice, `en-US-AriaNeural`, for every positive sample. A region-tagged
> `WW_LANG` (for example `en-GB`) instead selects the first entry in edge-tts's table
> for that region (`en-GB-LibbyNeural` for `en-GB`) — one fixed voice either way, chosen
> by the installed TTS plugin, not by this notebook. To vary the voice, install more
> than one OVOS TTS plugin (for example `ovos-tts-plugin-google-tx` alongside
> `ovos-tts-plugin-edge-tts`) so wakeforge's random per-sample plugin choice has more
> than one voice to draw from; to pin one exact voice, install only the OVOS TTS
> plugin for that voice (for example a single phoonnx voice package) and no other.

> **Existing dataset:** set `DATASET_DIR` to a dataset directory built by
> `synthetic_ww_dataset.ipynb` to skip dataset generation entirely and train on
> it instead (see Cell 5).


In [ ]:
import os

# ── Wake phrase — THE ONE THING YOU NEED TO CHANGE ────────────────────────────
WAKE_WORD        = os.environ.get("WAKE_WORD",        "hey jarvis")

# ── Output location ───────────────────────────────────────────────────────────
OUTPUT_DIR       = os.environ.get("OUTPUT_DIR",       "./ww_output")

# ── Existing dataset (optional) ───────────────────────────────────────────────
# A dataset directory built by synthetic_ww_dataset.ipynb: a metadata.csv of
# "path,label" rows beside positives/ and negatives/ audio. When set, Cell 5
# skips its own dataset generation and trains on it instead.
DATASET_DIR      = os.environ.get("DATASET_DIR",      "")

# ── Model ─────────────────────────────────────────────────────────────────────
TIER             = os.environ.get("TIER",             "small")
# SMOKE already set EPOCHS via papermill injection; only fall back to the env
# var / 50-epoch default on a real (non-SMOKE) run, so the smoke path keeps
# papermill's EPOCHS instead of it being clobbered back up to 50.
EPOCHS           = EPOCHS if SMOKE else int(os.environ.get("EPOCHS", "50"))
BATCH_SIZE       = int(os.environ.get("BATCH_SIZE",   "16"))
DEVICE           = os.environ.get("DEVICE",           "auto")
SEED             = int(os.environ.get("SEED",         "42"))

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE       = int(os.environ.get("N_POSITIVE",   "500"))
LANG             = os.environ.get("WW_LANG",          "en")  # not "LANG": that name collides with the POSIX locale env var
ADVERSARIAL      = os.environ.get("ADVERSARIAL",      "true").lower() == "true"
DOWNLOAD_AUGMENT = os.environ.get("DOWNLOAD_AUGMENT", "true").lower() == "true"
REUSE_DATASET    = os.environ.get("REUSE_DATASET",    "true").lower() == "true"

# ── MLflow (optional — leave blank to skip) ───────────────────────────────────
MLFLOW_URI       = os.environ.get("MLFLOW_URI",       "")
MLFLOW_SECRET    = os.environ.get("MLFLOW_SECRET",    "MLFLOW_TOKEN")

if SMOKE:
    # Shrink the run to N_SAMPLES synthetic positives, a fixed fast wake word,
    # and no augmentation downloads (background noise/music/RIR corpora).
    if not DATASET_DIR:
        # Only fabricate a wake phrase for the notebook's own synthetic smoke
        # dataset. A supplied DATASET_DIR carries its own wake word (built by
        # synthetic_ww_dataset.ipynb); overriding WAKE_WORD here would train
        # on that dataset's audio while naming and configuring the model for
        # a different phrase.
        WAKE_WORD    = SMOKE_WAKE_WORD.replace("_", " ")
    N_POSITIVE       = N_SAMPLES
    DOWNLOAD_AUGMENT = False

print(f"Wake word : {WAKE_WORD!r}")
print(f"Tier      : {TIER}  |  Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  Device: {DEVICE}")
print(f"Positives : {N_POSITIVE}  |  Lang: {LANG}  |  Augmentation: {DOWNLOAD_AUGMENT}")
if DATASET_DIR:
    print(f"Dataset   : reusing {DATASET_DIR} (dataset generation skipped)")


## Cell 3 — Install dependencies

Installs the scientific stack plus `wakeforge` with two extras:

- `[datagen]` — OVOS TTS plugin (edge-tts) and the HuggingFace `datasets` library
  used to download negative speech audio.
- `[torchcodec]` — audio codec backend required to decode MP3/AAC files from the
  HuggingFace datasets.

`wakeforge` is on PyPI under the alpha pre-release channel, so this cell
installs it with `pip install --pre`; if the install still fails, re-run this
cell after checking your network connection.

> **Expected runtime:** not measured; on the order of a few minutes on Kaggle
> when packages are not pre-cached.

In [ ]:
import sys, os

%pip install --quiet --pre "torch>=2.14.0" "torchaudio>=2.4" "onnx>=1.22.0" "onnxruntime>=1.30.0" "numpy>=2.0" "soundfile>=0.14.0" "librosa>=0.10.2" "scikit-learn>=1.9.0" "matplotlib>=3.11.1" "tqdm>=4.66" "click>=8.1"

# wakeforge (imported as ww_trainer) with data-generation and audio-codec extras.
%pip install --quiet --pre "wakeforge[datagen,torchcodec]>=0.4.1a3"

# The OVOS wake-word plugin that loads wakeforge's exported ONNX pair at
# inference time (Cell 10 loads the trained model through it).
%pip install --quiet --pre "ovos-ww-plugin-wakeforge>=0.0.4a1"

# Detect running platform
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "colab"      if "google.colab" in sys.modules else
    "paperspace" if os.path.exists("/notebooks")  else
    "local"
)

try:
    import ww_trainer
except ImportError:
    raise ImportError(
        "wakeforge is not installed (imported as ww_trainer). Run the install "
        "cell above (Cell 3) and re-run this cell. If PyPI installation failed "
        "(e.g. a network issue), check the %pip output above for the reason."
    )

print(f"Platform  : {_platform}")
print(f"ww_trainer: {ww_trainer.__version__}")
print(f"Python    : {sys.version.split()[0]}")


## Cell 4 — Platform setup (Colab Drive + Kaggle Secrets)

This cell handles two platform-specific tasks:

**On Google Colab:** mounts Google Drive and redirects `OUTPUT_DIR` to
`/content/drive/MyDrive/ww_output` so output files survive session resets.
You will be prompted once to authorise Drive access.

**On Kaggle:** reads the MLflow token from Kaggle Secrets (if configured) and injects
it as an environment variable.  Training works fine if no secret exists — this is
silently skipped.

**On any platform:** applies `MLFLOW_URI` if set in Cell 2.

> MLflow is entirely optional.  Skipping it has no effect on the trained model.

In [ ]:
import os

# ── Colab: mount Drive for persistent storage ─────────────────────────────────
if _platform == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/ww_output"
        print(f"Colab: outputs redirected to {OUTPUT_DIR} (survives session resets)")
    except Exception as e:
        print(f"Drive mount skipped ({e}). Outputs will be lost on session reset.")

# ── Kaggle: inject MLflow token from Secrets ──────────────────────────────────
if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"Kaggle: MLflow token injected from Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"Kaggle: no Secret '{MLFLOW_SECRET}' found — MLflow disabled. ({e})")

# ── Any platform: apply MLflow URI if provided ────────────────────────────────
if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    print(f"MLflow URI: {MLFLOW_URI}")
else:
    print("MLflow URI not set — experiment tracking disabled (training still works fine).")


## Cell 5 — Generate the training dataset

This cell builds the labelled audio dataset your model will train on.

**`DATASET_DIR` set (optional):** when it points at a dataset directory built
by `synthetic_ww_dataset.ipynb` — a `metadata.csv` of `path,label` rows beside
`positives/` and `negatives/` audio — this cell skips its own dataset
generation entirely, splits that CSV into a train/test set, and trains on it.

**Otherwise, what happens inside:**

1. **Positives** — edge-tts synthesises `N_POSITIVE` utterances of your wake phrase in
   multiple voices, speeds, and pitches.  Each is labelled `1` ("wake word").
2. **Negatives** — short speech clips that do *not* contain the phrase are downloaded
   from public HuggingFace datasets.  Each is labelled `0` ("not wake word").
3. **Adversarial negatives** (when `ADVERSARIAL=true`) — phonetically-similar phrases
   ("hay janice", "hey jarvi") are synthesised as hard negatives, making the model
   more robust to near-misses.
4. **Augmentation audio** (when `DOWNLOAD_AUGMENT=true`) — background noise, music, and
   room impulse responses are downloaded.  They are mixed in at training time to simulate
   real-world listening conditions.

**`REUSE_DATASET=true` (default):** if the dataset already exists at
`OUTPUT_DIR/dataset/`, this cell completes instantly — no re-downloads.
Re-run the whole notebook freely without waiting for downloads every time.

> **Expected runtime:** not measured; mostly HuggingFace downloads, so on the
> order of tens of minutes on Kaggle. Set `DOWNLOAD_AUGMENT=false` to cut this
> down substantially (slight accuracy trade-off).

> **Troubleshooting:**
> - *`ModuleNotFoundError: datasets`* — re-run Cell 3 (install did not complete).
> - *`AssertionError: Only X.X GB free`* — free up disk in the Kaggle Output tab.


In [ ]:
import json
import random
import shutil
from pathlib import Path
from ww_trainer.datagen import (
    DatagenConfig, DatagenResult, normalize_wake_word,
    read_metadata_csv, run_datagen_pipeline, write_metadata_csv,
)

# ── Disk space guard ──────────────────────────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > 3, (
    f"Only {free_gb:.1f} GB free — need at least 3 GB. "
    "Free up space or set DOWNLOAD_AUGMENT=false to reduce requirements."
)
print(f"Disk free: {free_gb:.1f} GB")

# ── Dataset paths ─────────────────────────────────────────────────────────────
dataset_dir        = Path(OUTPUT_DIR) / "dataset"
train_csv_expected = dataset_dir / "train" / "metadata.csv"
test_csv_expected  = dataset_dir / "test"  / "metadata.csv"

if DATASET_DIR:
    # An existing dataset built by synthetic_ww_dataset.ipynb: a flat
    # "path,label" metadata.csv beside positives/ and negatives/ audio. Split
    # it into the train/test layout train_from_wakeword() expects, instead of
    # generating a dataset of our own.
    src_dir = Path(DATASET_DIR).expanduser().resolve()
    src_metadata = src_dir / "metadata.csv"
    assert src_metadata.exists(), f"No metadata.csv under DATASET_DIR={src_dir}"
    raw_entries = list(read_metadata_csv(src_metadata))
    assert raw_entries, f"{src_metadata} has no rows"

    # metadata.csv records paths relative to its own directory (so the
    # dataset can be moved -- for example uploaded to Kaggle -- and still
    # resolve); ww_trainer reads every path verbatim, so resolve each one
    # against src_dir before writing the split CSVs handed to training.
    def _resolve_row(entry):
        raw_path, label = entry
        candidate = Path(raw_path)
        resolved = candidate if candidate.is_absolute() else src_dir / candidate
        assert resolved.exists(), (
            f"metadata.csv row {raw_path!r} does not resolve to an existing "
            f"file under DATASET_DIR={src_dir} (resolved: {resolved})"
        )
        return (str(resolved), label)

    entries = [_resolve_row(e) for e in raw_entries]
    random.Random(SEED).shuffle(entries)
    n_test = max(1, int(len(entries) * 0.2))
    test_entries, train_entries = entries[:n_test], entries[n_test:]
    write_metadata_csv(train_csv_expected, train_entries)
    write_metadata_csv(test_csv_expected, test_entries)
    _dr = DatagenResult(
        train_csv=train_csv_expected,
        test_csv=test_csv_expected,
        positives_dir=src_dir / "positives",
        negatives_dir=src_dir / "negatives",
    )
    print(f"Using existing dataset: {src_dir}")
    print(f"  {len(train_entries)} train rows, {len(test_entries)} test rows")

    # The wake word this dataset was built for -- not the SMOKE default
    # or the WAKE_WORD fallback -- names the model. dataset.json (written
    # by synthetic_ww_dataset.ipynb) is the source of truth; when it is
    # absent, the *_adversarials.txt filename encodes the same phrase.
    _dataset_json = src_dir / "dataset.json"
    if _dataset_json.exists():
        _dataset_wake_word = json.loads(_dataset_json.read_text())["wake_word"]
    else:
        _adv_files = sorted(src_dir.glob("*_adversarials.txt"))
        assert _adv_files, (
            f"DATASET_DIR={src_dir} has neither dataset.json nor "
            "*_adversarials.txt; cannot determine the wake word it was built for"
        )
        _dataset_wake_word = _adv_files[0].name[: -len("_adversarials.txt")].replace("_", " ")
    if "WAKE_WORD" not in os.environ:
        WAKE_WORD = _dataset_wake_word
        print(f"Wake word  : {WAKE_WORD!r} (from DATASET_DIR)")
    elif WAKE_WORD != _dataset_wake_word:
        raise ValueError(
            f"WAKE_WORD={WAKE_WORD!r} does not match the dataset's wake word "
            f"{_dataset_wake_word!r} recorded at {src_dir}; unset WAKE_WORD or "
            "point DATASET_DIR at a dataset built for this phrase"
        )
elif REUSE_DATASET and train_csv_expected.exists() and test_csv_expected.exists():
    # Reconstruct result object from existing paths (no downloads)
    print(f"Reusing existing dataset at {dataset_dir}")
    slug = normalize_wake_word(WAKE_WORD)
    _dr = DatagenResult(
        train_csv=train_csv_expected,
        test_csv=test_csv_expected,
        positives_dir=dataset_dir / slug / "positives",
        negatives_dir=dataset_dir / slug / "negatives",
        bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
        music_dir=dataset_dir / "augmentation" / "music",
        rir_dir=dataset_dir / "augmentation" / "rir",
    )
else:
    print(f"Generating dataset for {WAKE_WORD!r} — this may take a while "
          f"(mostly HuggingFace downloads)...")
    if SMOKE:
        # The smoke path uses a few generated noise clips as negatives.
        # wakeforge reuses a negatives folder that already holds enough WAV
        # files, so no negative corpus is downloaded
        # (https://github.com/TigreGotico/wakeforge/issues/42).
        import numpy as np
        import soundfile as sf
        from ww_trainer.datagen import NEGATIVE_DATASETS
        _rng = np.random.default_rng(SEED)
        for _category in ("general", "speech"):
            for _ds_id in NEGATIVE_DATASETS[_category]:
                _neg_dir = dataset_dir / "negatives" / _ds_id.split("/")[-1]
                _neg_dir.mkdir(parents=True, exist_ok=True)
                for _i in range(N_POSITIVE):
                    _noise = (_rng.standard_normal(16000) * 0.05).astype(np.float32)
                    sf.write(_neg_dir / f"smoke_noise_{_i:03d}.wav", _noise, 16000)
    _dr = run_datagen_pipeline(DatagenConfig(
        wake_word=WAKE_WORD,
        output_dir=dataset_dir,
        n_positive=N_POSITIVE,
        max_negative=N_POSITIVE if SMOKE else None,  # bound the negative pull to the smoke size
        lang=LANG,
        adversarial=ADVERSARIAL,
        vad_trim=True,
        download_augmentation=DOWNLOAD_AUGMENT,
        seed=SEED,
    ))

print(f"Train CSV : {_dr.train_csv}")
print(f"Test CSV  : {_dr.test_csv}")
if _dr.bg_noise_dir and Path(_dr.bg_noise_dir).exists():
    print(f"BG noise  : {_dr.bg_noise_dir}")
print("Dataset ready.")


## Cell 6 — Train

Calls `train_from_wakeword()`, which:

1. Detects and reuses the dataset built in Cell 5 (`reuse_dataset=True`).
2. Instantiates the model architecture specified by `TIER`.
3. Trains for `EPOCHS` epochs, saving the best checkpoint by F1 score.
4. Exports two ONNX files: the audio feature extractor and the classifier head.

**Outputs land in `OUTPUT_DIR/model/`.**

The augmentation directories prepared in Cell 5 (background noise, music, reverb)
are picked up automatically — no extra arguments are needed here.

> **Expected runtime:** not measured; depends on `N_POSITIVE`, `EPOCHS`, `TIER`
> and whether augmentation audio was downloaded.

> **Troubleshooting:**
> - *CUDA out of memory* — reduce `BATCH_SIZE` to `8` in Cell 2, re-run from Cell 6.
> - *F1 stays very low (< 0.5)* — increase `N_POSITIVE` to 1000 and re-run from
>   Cell 5 with `REUSE_DATASET=false`.  More data usually fixes this.
> - *Kaggle session timeout (Kaggle's free-tier GPU sessions have a 9-hour
>   limit)* — `train_from_wakeword()` has no checkpoint-resume; re-running
>   this cell retrains from epoch 0. The dataset built in Cell 5 is reused
>   instantly (`REUSE_DATASET=true`), so only training time is lost. Lower
>   `EPOCHS` or `N_POSITIVE` to fit one session.


In [ ]:
from ww_trainer.quickstart import train_from_wakeword

print(f"Training: wake_word={WAKE_WORD!r}  tier={TIER!r}  epochs={EPOCHS}  "
      f"batch_size={BATCH_SIZE}  device={DEVICE!r}")

# Note: augmentation dirs from Cell 5 are picked up automatically
# when reuse_dataset=True — no need to pass them explicitly here.
result = train_from_wakeword(
    WAKE_WORD,
    OUTPUT_DIR,
    tier=TIER,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=(0.05 if SMOKE else 5e-4),  # SMOKE trains one epoch; a higher
    # learning rate is needed for that one epoch to move the model
    # enough that the positive clip separates from the negative median
    # (Cell 8) by more than chance.
    device=DEVICE,
    seed=SEED,
    n_positive=N_POSITIVE,
    lang=LANG,
    adversarial=ADVERSARIAL,
    download_augmentation=DOWNLOAD_AUGMENT,
    reuse_dataset=True,   # dataset was built in Cell 5
)

print("\nTraining complete.")
print(f"  best_onnx_path  : {result.best_onnx_path}")
print(f"  best_model_path : {result.best_model_path}")
print(f"  metrics         : {result.metrics}")


## Cell 7 — ONNX export check

Verifies that both required ONNX files were written successfully.

**Why two files?**

- `best_f1_featurizer.onnx` — converts raw 16 kHz audio to a feature vector
  (MFCC, filterbank, etc.).
- `best_f1.onnx` — takes that feature vector and outputs a confidence score in `[0, 1]`.

Both must be copied to your device together.  At inference time they are chained:

```
raw audio → featurizer.onnx → features → classifier.onnx → score
```

Neither file requires PyTorch at inference time — only `onnxruntime` and `numpy`.

If either file is missing, check the training log above for errors.

In [ ]:
from pathlib import Path

model_dir  = Path(OUTPUT_DIR) / "model"
_feat_onnx = model_dir / "best_f1_featurizer.onnx"
_head_onnx = model_dir / "best_f1.onnx"

print("ONNX files:")
for label, path in [("featurizer", _feat_onnx), ("classifier", _head_onnx)]:
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  OK      {label}: {path.name}  ({size_kb:.0f} KB)")
    else:
        print(f"  MISSING {label}: {path}")

assert _feat_onnx.exists() and _head_onnx.exists(), (
    "One or both ONNX files are missing. Check the training output above for errors."
)
print("\nBoth ONNX files present — ready for deployment.")


## Cell 8 — Inference sanity check

Loads both ONNX files with `OnnxWakeWordInferencer` — the same class used at runtime
in OVOS — and scores one positive sample plus several negative samples from the
dataset.

This test uses only `onnxruntime` and `numpy`; no PyTorch is needed.

**What is asserted:** the positive clip's score must exceed the median score of
the negative clips by more than `0.02` (`DETECTION_MARGIN`). This is a
threshold-free property — it does not depend on picking a confidence cutoff like
`0.5` — but a bare `>` on its own reads noise on a `SMOKE` model: at the default
learning rate, one epoch on a handful of samples leaves every score close to 0.5
(measured: 0.4907 against a median negative of 0.4478, a margin of 0.0429, on a
run of this cell before Cell 6's learning-rate change). Cell 6 raises the
learning rate under `SMOKE` so one epoch moves the model further, and this cell
scores the positive clip against a training-split clip instead of a held-out
one, since the model has actually been trained on it; a measured run with both
changes scored 0.9998 against a median negative of 0.9720, a margin of 0.0278.
`0.02` is below both measured margins. A confident, deployment-ready detection
(score ≥ 0.8, or the OVOS plugin's `found_wake_word()` firing) is not asserted
here; Cell 10 runs that same check through the OVOS plugin and reports it
without asserting it, for the same reason.

**Interpreting the printed score:**

| Score | Meaning |
|-------|---------|
| ≥ 0.8 | Model is confident — ready to deploy |
| 0.5–0.8 | Borderline — consider more epochs or more data |
| < 0.5 | Low confidence — increase `N_POSITIVE` to 1000, `EPOCHS` to 100, re-run |

The default detection threshold in the OVOS plugin is `0.5` (configurable).

In [ ]:
import csv, numpy as np, soundfile as sf
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

inferencer = OnnxWakeWordInferencer(str(_feat_onnx), str(_head_onnx))
print("OnnxWakeWordInferencer loaded (onnxruntime only — no PyTorch at runtime).")


def _score_clip(path: str) -> float:
    wav_np, sr = sf.read(path, dtype="float32", always_2d=True)
    assert sr == 16000, f"expected 16 kHz audio, got {sr} Hz: {path}"
    wav_np = wav_np.mean(axis=1).astype(np.float32)  # downmix to mono
    return float(inferencer.infer(wav_np))


def _sample_paths(csv_path, label: str, limit: int | None = None) -> list[str]:
    paths = []
    with open(csv_path) as f:
        for row in csv.reader(f):
            if len(row) >= 2 and row[1].strip() == label and Path(row[0]).exists():
                paths.append(row[0])
                if limit and len(paths) >= limit:
                    break
    return paths


# The positive clip is drawn from the training split, not the held-out
# test split: the model has actually trained on it, where a held-out
# clip sits too close to the untrained 0.5 baseline for the margin below
# to mean anything. Fall back to the test split if training has none.
_pos_path = next(iter(_sample_paths(result.train_csv, "1")), None) or \
    next(iter(_sample_paths(result.test_csv, "1")), None)
_neg_paths = _sample_paths(result.train_csv, "0", limit=10) or \
    _sample_paths(result.test_csv, "0", limit=10)

assert _pos_path, "no positive clip found in either split — cannot check detection"
assert _neg_paths, "no negative clip found in either split — cannot check detection"

_score = _score_clip(_pos_path)
_neg_scores = [_score_clip(p) for p in _neg_paths]
_median_neg = float(np.median(_neg_scores))
_margin = _score - _median_neg

_verdict = (
    "PASS — model confident"                     if _score >= 0.8 else
    "BORDERLINE — consider more data or epochs"  if _score >= 0.5 else
    "LOW — needs more data or training"
)
print(f"\nPositive sample : {Path(_pos_path).name}")
print(f"Confidence score: {_score:.4f}  ({_verdict})")
print(f"Negative scores : n={len(_neg_scores)}  median={_median_neg:.4f}")
print(f"Margin          : {_margin:.4f}")

# A stated, checked margin: 0.4907 against 0.4478 (margin 0.0429) at the
# default learning rate and a held-out positive; 0.9998 against 0.9720
# (margin 0.0278) after Cell 6 raised the SMOKE learning rate and this
# cell moved to a training-split positive. 0.02 is below both measured
# margins; a bare > 0 comparison reads noise on an under-trained SMOKE
# model, this does not.
DETECTION_MARGIN = 0.02
assert _margin > DETECTION_MARGIN, (
    f"positive score {_score:.4f} exceeds the median negative score "
    f"{_median_neg:.4f} by only {_margin:.4f}, at or below the "
    f"{DETECTION_MARGIN} margin this check requires; the model did not "
    "learn to separate the wake word from non-wake-word audio"
)

## Cell 9 — Ship it to your assistant

The two ONNX files are everything needed.  Here is how to get them from this session
onto your device and into OpenVoiceOS.

### Step 1 — Download the files

**On Kaggle:** go to the **Output** tab (right sidebar) → navigate to
`ww_output/model/` → download `best_f1.onnx` and `best_f1_featurizer.onnx`.

**On Colab:** both files are in your Google Drive under
`MyDrive/ww_output/model/` — accessible from any browser.

### Step 2 — Copy to your OVOS device

```bash
scp best_f1_featurizer.onnx best_f1.onnx  \
    ovos@mydevice:~/.local/share/mycroft/precise/
```

### Step 3 — Configure OVOS (ovos-ww-plugin-wakeforge)

Install the plugin if not already present:

```bash
pip install --pre 'ovos-ww-plugin-wakeforge'
```

Add the following to `~/.config/mycroft/mycroft.conf` on your device (replace
`hey_jarvis` and the paths with your actual phrase and locations):

```json
{
  "listener": {
    "wake_word": "hey_jarvis"
  },
  "hotwords": {
    "hey_jarvis": {
      "module": "ovos-ww-plugin-wakeforge",
      "featurizer": "~/.local/share/mycroft/precise/best_f1_featurizer.onnx",
      "model": "~/.local/share/mycroft/precise/best_f1.onnx",
      "threshold": 0.5
    }
  }
}
```

The featurizer and model paths above match where Step 2 copied the
files; point them at wherever the files actually landed on your device.

- `threshold` — detection threshold `[0, 1]`.  `0.5` is the default.  \
  Raise toward `0.7` to reduce false activations; lower toward `0.3` for higher recall.
- `patience` — consecutive above-threshold frames required to fire.  Defaults to `3`.

### Step 4 — Restart and test

```bash
systemctl --user restart ovos-listener
journalctl --user -u ovos-listener -f   # watch for "WW activated"
```

The next two cells load the trained model through this same plugin class and
run it over one positive clip from the test split, as a check that the config
above is correct before it is copied to a device.

---

### Want a better model?

| Goal | What to change |
|------|----------------|
| Fewer false activations | Raise `EPOCHS` to 100; add real far-field recordings |
| Works well in noisy rooms | Ensure `DOWNLOAD_AUGMENT=true`; raise `N_POSITIVE` to 1000+ |
| Even smaller file (< 100 KB) | Use `TIER=micro` |
| Targeting ESP32 / MCU | See wakeforge's [`docs/guides/embedded.md`](https://github.com/TigreGotico/wakeforge/blob/dev/docs/guides/embedded.md) for the C export path |
| Compare architectures and losses | See wakeforge's [`notebooks/kaggle_experiments.ipynb`](https://github.com/TigreGotico/wakeforge/blob/dev/notebooks/kaggle_experiments.ipynb) |


In [ ]:
import json
from pathlib import Path

_slug      = WAKE_WORD.lower().replace(" ", "_")
_model_dir = Path(OUTPUT_DIR) / "model"
_feat_path = _model_dir / "best_f1_featurizer.onnx"
_head_path = _model_dir / "best_f1.onnx"

print("=" * 60)
print(f"Wake word : {WAKE_WORD!r}")
print(f"Tier      : {TIER}")
print(f"Metrics   : {result.metrics}")
print()
print("Output ONNX files:")
for f in sorted(_model_dir.glob("*.onnx")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")
print("=" * 60)

# OVOS config snippet
_ovos_hotword_cfg = {
    "module": "ovos-ww-plugin-wakeforge",
    "featurizer": str(_feat_path),  # path under OUTPUT_DIR; copy it wherever the device reads models from
    "model": str(_head_path),       # path under OUTPUT_DIR; copy it wherever the device reads models from
    "threshold": 0.5,
}
_ovos_cfg = {
    "listener": {"wake_word": _slug},
    "hotwords": {_slug: _ovos_hotword_cfg},
}
print()
print("OVOS mycroft.conf snippet (update paths to your device):")
print(json.dumps(_ovos_cfg, indent=2))


## Cell 10 — Load through the OVOS plugin

Loads the same two ONNX files through `ovos-ww-plugin-wakeforge`'s
`WakeForgeHotwordPlugin` — the class OVOS actually instantiates from the
`hotwords` config in Step 3 above — and runs it over one positive clip. This
checks that the config snippet above loads and scores audio; a model trained
on `N_SAMPLES` smoke-path examples rarely reaches a confident detection, so no
confident-detection outcome is asserted here, only that the plugin loads its
config and runs.


In [ ]:
import soundfile as sf
from ovos_ww_plugin_wakeforge import WakeForgeHotwordPlugin

plugin = WakeForgeHotwordPlugin(key_phrase=_slug, config=_ovos_hotword_cfg)
print(f"WakeForgeHotwordPlugin loaded for {_slug!r} "
      f"(threshold={plugin.threshold}, streaming={plugin.streaming}).")

# _pos_path was already asserted to exist in Cell 8, so this always has
# a real positive clip to feed the plugin.
_pcm, _pcm_sr = sf.read(_pos_path, dtype="int16", always_2d=True)
assert _pcm_sr == 16000, f"expected 16 kHz audio, got {_pcm_sr} Hz: {_pos_path}"
_pcm_bytes = _pcm[:, 0].tobytes()  # mono int16 PCM, as OVOS's mic feed delivers it

plugin.update(_pcm_bytes)
_triggered = plugin.found_wake_word()
print(f"Fed {len(_pcm_bytes) // 2} samples of {Path(_pos_path).name} "
      f"through the plugin — found_wake_word()={_triggered}")

Licensed under Apache-2.0.